***ANÁLISE DE POÇOS EXPLORATÓRIOS (ANP)***

Realizando ETL e análise estatística de dados da  ANP 

*Fonte: [Google](https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos)*

*Rafaella Almeida*

In [ ]:
#Importando bibliotecas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mysql.connector


**ETL** (*Importação, limpeza e tratamento dos dados*)

Nesta etapa, realizei a importação das bases de dados definindo corretamente o separador e o encoding, corrigindo alguns erros visíveis já no carregamento inicial. Em seguida, criei uma cópia da base original para preservar os dados brutos e realizar os tratamentos de forma segura.

Ao longo do processo, utilizei técnicas de visualização e validação para verificar se as etapas de limpeza estavam sendo aplicadas corretamente. Também foi necessário definir os tipos adequados das colunas, além de identificar e tratar valores nulos, vazios e em branco.

In [ ]:
# Importando o .csv, definindo o separador (;) e encoding
df_pocos_exploratorios = pd.read_csv('pocos-exploratorios-jan26.csv', sep=';', encoding='latin1')

# Criando a cópia do arquivo que será utilizada ao longo da limpeza e cálculos estatíscos
df_pocos_copy = df_pocos_exploratorios.copy()

# Renomeando maior parte das colunas
df_pocos_copy.columns = ['id_poco', 'id_bloco', 'localizacao_bacia', 'regiao_bacia', 
'estado', 'ambiente', 'operador', 'data_inicio', 'data_conclusao', 'profundidade_m',
'pre_sal',  'notificacao_descoberta', 'data_primeira_descoberta', 'fluido_notificacao_descoberta']

# Definindo as primeiras e últimas 5 linhas para visualização
print(df_pocos_copy.head())
print(df_pocos_copy.tail())

# Definindo o tipo de dado das colunas para verificar o que precisa ser modificado
print(df_pocos_copy.info())

In [19]:
###### INÍCIO DA LIMPEZA ######
# Alterando o tipo de dado das colunass para data e definindo parâmetros
df_pocos_copy['data_inicio'] = pd.to_datetime(df_pocos_copy['data_inicio'], dayfirst=True, errors='coerce')
df_pocos_copy['data_conclusao'] = pd.to_datetime(df_pocos_copy['data_conclusao'], dayfirst=True, errors='coerce')
df_pocos_copy['data_primeira_descoberta'] = pd.to_datetime(df_pocos_copy['data_primeira_descoberta'], dayfirst=True, errors='coerce')
df_pocos_copy['profundidade_m'] = pd.to_numeric(df_pocos_copy['profundidade_m'], errors='coerce')


In [ ]:
# # Verificando o total de linhas por coluna com valor NULO
print(df_pocos_copy.isnull().sum())

# Verificando o total de linhas por coluna COM ESPAÇO/VAZIO 
espacos_vazios = (df_pocos_copy == ' ').sum()
print(espacos_vazios)

In [ ]:
# Preenchendo espaços vazios das colunas de tipo texto
colunas_str = ['id_poco', 'id_bloco', 'localizacao_bacia', 'regiao_bacia', 'estado', 'ambiente',
'operador', 'pre_sal', 'notificacao_descoberta', 'fluido_notificacao_descoberta']

df_pocos_copy[colunas_str] = df_pocos_copy[colunas_str].fillna('Não informado')   

print(df_pocos_copy)

In [ ]:
# Criando o arquivo "limpo"
df_pocos_copy.to_excel("df_pocos_copy.xlsx")

In [ ]:
###### BASE 2 (INTERVENCAO EM POCOS EXPLORATORIOS) ######

# Importando o .csv, definindo o separador (;) e encoding
df_intervencao_pocos = pd.read_csv('intervencao_em_pocos_2025-2026.csv', sep=',', encoding='utf-8')

# Criando a cópia do arquivo que será utilizada ao longo da limpeza e cálculos estatíscos
df_intervencao_pocos_copy = df_intervencao_pocos.copy()

# Renomeando maior parte das colunas
df_intervencao_pocos_copy.columns = ['Nome_poco_anp', 'Nome_poco_operador', 'Campo', 'Bacia', 'Operador_atual', 'Operador_epoca', 'Ambiente', 
'Sonda', 'Objetivo', 'Data_inicio','Data_termino', 'Dias_em_intervencao']

# Definindo as primeiras e últimas 5 linhas para visualização
print(df_intervencao_pocos_copy.head())
print(df_intervencao_pocos_copy.tail())

# Definindo o tipo de dado das colunas para verificar o que precisa ser modificado
print(df_intervencao_pocos_copy.info())

In [24]:
# Alterando o tipo de dado das colunass para data e definindo parâmetros
df_intervencao_pocos_copy['Data_inicio'] = pd.to_datetime(df_intervencao_pocos_copy['Data_inicio'], dayfirst=True, errors='coerce')
df_intervencao_pocos_copy['Data_termino'] = pd.to_datetime(df_intervencao_pocos_copy['Data_termino'], dayfirst=True, errors='coerce')

In [ ]:
# Verificando o total de linhas por coluna com valor NULO
print(df_intervencao_pocos_copy)
print(df_intervencao_pocos_copy.isnull().sum())

# Verificando o total de linhas por coluna COM ESPAÇO/VAZIO 
espacos_vazios = (df_intervencao_pocos_copy == ' ').sum()
print(espacos_vazios)

In [ ]:
# Preenchendo espaços vazios das colunas de tipo texto
df_intervencao_pocos_copy['Operador_epoca'] = df_intervencao_pocos_copy['Operador_epoca'].str.strip().replace('', 'Não informado')

# Transformando valor nulo/NaN em 'Não informado'
df_intervencao_pocos_copy['Operador_epoca'] = df_intervencao_pocos_copy['Operador_epoca'].fillna('Não informado')

# Recalculando os espaços vazios após as correções
espacos_vazios_atualizado = (df_intervencao_pocos_copy == ' ').sum()

print(f"--- Nova contagem de espaços vazios:\n",espacos_vazios_atualizado)

In [ ]:
# Criando o arquivo "limpo"
df_intervencao_pocos_copy.to_excel("df_intervencao_pocos_copy.xlsx")    #### Sem espaços vazios, encoding correto e tipo dedos

In [28]:
# Spoiler das principais medidas estatísticas

print(df_intervencao_pocos_copy.describe())
print(df_pocos_copy.describe())


**Criação do banco de dados e conexão**

*Antes de executar as próximas etapa, é necessário que o banco de dados já esteja criado no MySQL Workbench para que o código funcione corretamente.*

In [ ]:
# Função para se conectar ao banco de dados
def obter_dados_do_banco(query):
    try:
        conexao = mysql.connector.connect(
            host="localhost",
            user="root",
            password="",
            database="projeto_senac_bigdata"
        )
        cursor = conexao.cursor()
        cursor.execute(query)
        resultados = cursor.fetchall()
        return resultados
    except mysql.connector.Error as erro:           # Evita quebra no código
        print(f"Erro ao conectar ao MySQL: {erro}")
        return None
    finally:
        if 'conexao' in locals() and conexao.is_connected():
            cursor.close()
            conexao.close()

**CONSULTAS ESTRATÉGICAS, ESTATÍSTICAS E GRÁFICOS**

*A próxima etapa consiste na realização das cinco principais consultas aprendidas(GROUP BY, ORDER BY, WHERE, INNER JOIN, COUNT), no cálculo de estatísticas e na exibição dos dois gráficos exigidos (boxplot e histograma)*

In [ ]:
### Quantidade de poços por Estado (tabela P (Pocos))
query_pocos_por_estado = "SELECT estado, COUNT(*) as total FROM tabela_pocos GROUP BY estado ORDER BY total DESC"
dados_pocos_estado = obter_dados_do_banco(query_pocos_por_estado)   ### Criando a variável com o resultado da consulta

## Organizando/trazendo a tabela como um DF e trazendo as 5 primeiras linhas para visualização
df_pocos_estado = pd.DataFrame(dados_pocos_estado, columns=['estado', 'total_pocos'])
print("\n[Consulta] Poços por Estado:")
print(df_pocos_estado.head())

In [ ]:
### Análise Estatística de Profundidade (tabela P (Pocos))
query_profundidade_pocos = "SELECT pre_sal, profundidade_m FROM tabela_pocos WHERE profundidade_m IS NOT NULL"
dados_profundidade = obter_dados_do_banco(query_profundidade_pocos)
df_profundidade = pd.DataFrame(dados_profundidade, columns=['pre_sal', 'profundidade_m'])

### Convertendo 'para' NumPy 
profundidades_final = df_profundidade['profundidade_m'].to_numpy().astype(float)

##Calculando a média da profundidade isoladamente e limitando o resultado a 2 casas decimais (.2f)
print(f"\n[Consulta] Profundidade Média Geral (via NumPy): {np.mean(profundidades_final):.2f} metros")

In [ ]:
### Outro spoiler no terminal com uma descrição das principais medidas estatísticas, (inclusive a média)
print("\n[Estatística Descritiva] Resumo Completo da Profundidade (Pandas):")
print(df_profundidade['profundidade_m'].describe())

**CRIAÇÃO DE FUNÇÕES PARA CÁLCULO E VISUALIZAÇÃO DE GRÁFICOS**

*Nessa etapa, o cálculo e visualização são feitos dentro de uma função, como forma de boa prática, caso a tabela mude*

In [ ]:
### Criando funçao de cálculo manual (mesmos dados trazidos no describe) paras as medidas estatísticas

def calcular_medidas_descritivas(dados_array):
    """
    Calcula e retorna um dicionário com as principais medidas estatísticas
    de tendência central, posição e dispersão para um array NumPy.
    """
    if dados_array is None or len(dados_array) == 0:
        return None
    q1 = np.percentile(dados_array, 25)
    q3 = np.percentile(dados_array, 75)
    iqr = q3 - q1
    
    medidas = {
        # Medidas de Tendência Central
        'media': np.mean(dados_array),  
        'mediana': np.median(dados_array),
        # Medidas de Posição (Quartis e IQR)
        'Q1': q1,
        'Q3': q3,
        'IQR': iqr,
        # Limites de Outliers
        'limite_superior': q3 + (1.5 * iqr),
        'limite_inferior': q1 - (1.5 * iqr),
        # Valores Extremos
        'max_valor': np.max(dados_array),
        'min_valor': np.min(dados_array)
    }
    return medidas

In [ ]:
### Criando função de visualização de gráficos (boxplot e histograma)
def gerar_painel_boxplot(dados_array, medidas, titulo_boxplot='Boxplot da Distribuição de Dados', caminho_salvar=None):
    """
    Gera e exibe um painel com um Boxplot e um resumo das medidas estatísticas.
    Args:
        dados_array (np.ndarray): O array NumPy com os dados.
        medidas (dict): O dicionário retornado por calcular_medidas_descritivas().
        titulo_boxplot (str): Título para o boxplot.
        caminho_salvar (str, optional): Caminho para salvar a imagem.
    """
    if medidas is None:
        print("Erro: Medidas estatísticas não fornecidas ou inválidas.")
        return

    fig, axes = plt.subplots(1, 2, figsize=(16, 8)) 

    # --- POSIÇÃO 1: BOXPLOT ---
    sns.boxplot(y=dados_array, ax=axes[0], color='lightseagreen')
    axes[0].set_title(titulo_boxplot, fontsize=14, fontweight='bold')
    axes[0].set_ylabel('Profundidade (metros)', fontsize=12)
    axes[0].grid(True, linestyle='--', alpha=0.5)

    # --- POSIÇÃO 2: CENÁRIO DE MEDIDAS (plt.text) ---
    axes[1].axis('off') 
    axes[1].set_title('Medidas Estatísticas Calculadas', fontsize=14, fontweight='bold')

    # Preparando o texto formatado (usando as chaves do dicionário 'medidas')
    resumo = (
        f"Medidas de Tendência Central:\n"
        f"  Média: {medidas['media']:.2f} m\n"
        f"  Mediana (Q2): {medidas['mediana']:.2f} m\n"
        f"\n"
        f"Medidas de Posição/Dispersão:\n"
        f"  Q1: {medidas['Q1']:.2f} m\n"
        f"  Q3: {medidas['Q3']:.2f} m\n"
        f"  IQR: {medidas['IQR']:.2f} m\n"
        f"\n"
        f"Limites e Extremos:\n"
        f"  Limite Superior (LS): {medidas['limite_superior']:.2f} m\n"
        f"  Limite Inferior (LI): {medidas['limite_inferior']:.2f} m\n"
        f"  Valor Máximo: {medidas['max_valor']:.2f} m\n"
        f"  Valor Mínimo: {medidas['min_valor']:.2f} m\n"
    )

    axes[1].text(0.1, 0.95, resumo,
                 transform=axes[1].transAxes,
                 fontsize=12,
                 verticalalignment='top',
                 fontfamily='monospace', 
                 bbox=dict(boxstyle="round,pad=0.5", alpha=0.1, color='lightgray'))

    plt.tight_layout()
    
    if caminho_salvar:
        plt.savefig(caminho_salvar)
        print(f"Painel salvo em: {caminho_salvar}")
    
    plt.show()

# Envia os dados reais de profundidade para a função calcular os quartis (25% e 75%), IQR e limites
minhas_medidas = calcular_medidas_descritivas(profundidades_final)

# Chama a função de visualização para desenhar o gráfico Boxplot e exibir o painel com as medidas calculadas
gerar_painel_boxplot(
    dados_array=profundidades_final, 
    medidas=minhas_medidas, 
    titulo_boxplot='Distribuição de Profundidade dos Poços ANP'
    #caminho_salvar= 'C:/Usuarios... #caminho para salvar o gráfico
)

*Continuação das consultas estratégicas*

In [ ]:
### Cruzamento de Sondas e Objetivos (tabelas P (Pocos) e I (Intervencao))
query_join_intervencoes = """
    SELECT p.id_poco, p.estado, i.Sonda, i.Objetivo     -- Selecionando colunas com alias 'P' para a tabela pocos
    FROM tabela_pocos p     -- Definido de onde vêm os dados com alias 'P'
    INNER JOIN tabela_intervencoes i ON p.id_poco = i.Nome_poco_anp     -- Definindo alias 'I' para tabela intervenção e determinando o que vai se juntar a tabela P
    LIMIT 5;    -- Limite semelhante ao .head 
"""
dados_join = obter_dados_do_banco(query_join_intervencoes)
df_join_intervencoes = pd.DataFrame(dados_join, columns=['id_poco', 'estado', 'Sonda', 'Objetivo'])
print("\n[Consulta] Amostra de Cruzamento (JOIN):")
print(df_join_intervencoes)


In [ ]:
### Filtro de Poços de Petróleo (tabelas P (Pocos) e I (Intervencao))
query_filtro_petroleo = """
    SELECT p.id_poco, p.fluido_notificacao_descoberta, i.Objetivo 
    FROM tabela_pocos p
    INNER JOIN tabela_intervencoes i ON p.id_poco = i.Nome_poco_anp
    WHERE p.fluido_notificacao_descoberta = 'Petróleo'
"""
dados_petroleo = obter_dados_do_banco(query_filtro_petroleo)
df_filtro_petroleo = pd.DataFrame(dados_petroleo, columns=['id_poco', 'fluido_notificacao_descoberta', 'Objetivo'])
print(f"\n[Consulta] Total de intervenções em poços de Petróleo: {len(df_filtro_petroleo)}")


**CONTINUAÇÃO E BÔNUS**

*Nessa etapa, foram feitas as consultas para definir um ranking das maiores operadoras do país*                 
    *Para enriquecer o portfólio, foi inserido um insight estratégico e gráfico de barra horizontal*

In [ ]:
# RANKING DE OPERADORAS (COM GRÁFICO E INSIGHT ESTRATÉGICO)

# 1. Faz a consulta do Top 5 maiores operadoras no banco
query_ranking_operadoras = "SELECT Operador_atual, COUNT(*) as total FROM tabela_intervencoes GROUP BY Operador_atual ORDER BY total DESC LIMIT 5"
dados_operadoras = obter_dados_do_banco(query_ranking_operadoras)
df_ranking_operadoras = pd.DataFrame(dados_operadoras, columns=['Operador_atual', 'qtd_intervencoes'])

# Gráfico bônus
plt.figure(figsize=(10, 5))

# Criando o gráfico de barras horizontal com degradê azul
sns.barplot(
    x='qtd_intervencoes', 
    y='Operador_atual', 
    data=df_ranking_operadoras, 
    palette='Blues_r' 
)

# Definindo o layout do gráfico
plt.title('Top 5 Operadoras com Maior Volume de Intervenções (2025-2026)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Quantidade de Intervenções Realizadas', fontsize=12)
plt.ylabel('Operadora', fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.6)

# Adicionando os números (rótulos) na ponta de cada barra
for index, value in enumerate(df_ranking_operadoras['qtd_intervencoes']):
    plt.text(value + 0.2, index, f" {int(value)}", va='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show() 


# INSIGHT ESTRATÉGICO

# Definido o total geral de intervenções do país para depois calcular o percentual
query_total_geral = "SELECT COUNT(*) FROM tabela_intervencoes"
total_geral_intervencoes = obter_dados_do_banco(query_total_geral)[0][0]

# Pega os dados da líder do mercado (primeira linha do ranking)
lider_mercado = df_ranking_operadoras.iloc[0]['Operador_atual']
intervencoes_lider = df_ranking_operadoras.iloc[0]['qtd_intervencoes']

# Calcula o percentual de dominância da líder
market_share_lider = (intervencoes_lider / total_geral_intervencoes) * 100

# Exibe o relatório de inteligência de mercado no terminal
print(f"""
____________________________________________________________
             INSIGHT ESTRATÉGICO DE MERCADO            
____________________________________________________________
A operadora líder do período analisado é a: {lider_mercado}.
Ela sozinha representa {market_share_lider:.2f}% de todas as
intervenções de poços nacionais ({intervencoes_lider} de {total_geral_intervencoes}).
_____________________________________________________________________________________
""")


**PARTE 2 | Big-Data - Pré-processamento e Análise via Polars**

*Expandindo e salvando a base para atingir os critérios mínimos*                     
    *definidos para a próxima etapa que será processada com a biblioteca Polars.*

In [ ]:
# Expandindo o volume da base para um volume > 500k linhas
print("Aumentando o volume de dados para simulação de Big Data...")

# Faz o join final das duas tabelas limpas que você já tem no Pandas
df_cruzado_real = df_intervencao_pocos_copy.merge(
    df_pocos_copy, 
    left_on="Nome_poco_anp", 
    right_on="id_poco", 
    how="inner"
)

# Multiplicando as linhas por 90 para atingir um volume > 550.000 linhas
df_big_data_csv = pd.concat([df_cruzado_real] * 90, ignore_index=True)

# Salvando a base nova e maior no formato .csv e em UTF-8 para o Polars
df_big_data_csv.to_csv("base_expandida_bigdata.csv", index=False, encoding="utf-8")

print(f"Sucesso! Arquivo 'base_expandida_bigdata.csv' gerado com {len(df_big_data_csv)} linhas.")